In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kano2012face")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "kano2012face_new_body.csv")
complete_path_2 = os.path.join(original_data_pathway, "kano2012face_new_face.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)


In [3]:
df1_list =df1[['group', 'spe', 'Subject', 'Face_allospecific', 'Body_allospecific',
       'Background_allospecific', 'Face_conspecific', 'Body_conspecific',
       'Background_conspecific']].values.tolist() 
df2_list = df2[['stimspe', 'group', 'spe', 'Subject', 'Eye_female', 'Nose_female',
       'Mouth_female', 'Periphery_female', 'Eye_male', 'Nose_male',
       'Mouth_male', 'Periphery_male', 'Eye_infant', 'Nose_infant',
       'Mouth_infant', 'Periphery_infant']].values.tolist()
combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(df1_list,df2_list)] 
df = pd.DataFrame(combined_lol, columns=['group', 'spe', 'participant', 'whole_body_Face_allospecific', 'whole_body_Body_allospecific',
       'whole_body_Background_allospecific', 'whole_body_Face_conspecific', 'whole_body_Body_conspecific',
       'whole_body_Background_conspecific', 'whole_body_stimspe', 'whole_body_group_2', 'spe_2', 'Subject_2', 
       'face_picture_Eye_female', 'face_picture_Nose_female',
       'face_picture_Mouth_female', 'face_picture_Periphery_female', 'face_picture_Eye_male', 'face_picture_Nose_male',
       'face_picture_Mouth_male', 'face_picture_Periphery_male', 'face_picture_Eye_infant', 'face_picture_Nose_infant',
       'face_picture_Mouth_infant', 'face_picture_Periphery_infant'])

In [4]:
df['study_id']="kano2012face"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')
# df.columns

complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True)

In [6]:
kano2012face_standardized=df[['study_id', 'participant','age_original','age_in_years', 'sex','species',
                              'whole_body_face_allospecific',
       'whole_body_body_allospecific', 'whole_body_background_allospecific',
       'whole_body_face_conspecific', 'whole_body_body_conspecific',
       'whole_body_background_conspecific',
       'face_picture_eye_female',
       'face_picture_nose_female', 'face_picture_mouth_female',
       'face_picture_periphery_female', 'face_picture_eye_male',
       'face_picture_nose_male', 'face_picture_mouth_male',
       'face_picture_periphery_male', 'face_picture_eye_infant',
       'face_picture_nose_infant', 'face_picture_mouth_infant',
       'face_picture_periphery_infant']]
comp_out_path_stand = os.path.join(out_pathway, 'kano2012face_standardized.csv')
kano2012face_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =kano2012face_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
kano2012face_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'kano2012face_glossary.csv')
kano2012face_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
